# AutoSemilla VQCC — verificación consolidada

Ejecuta pruebas del motor, búsqueda en los tres benchmarks, persistencia, ingesta de resultados y descubrimiento de inputs oficiales. La prueba sintética del adaptador NO es un PASS de competencia.

In [ ]:
from pathlib import Path
import os,sys,json,time,hashlib,subprocess,shutil,tempfile,platform
from datetime import datetime,timezone
REPO_URL="https://github.com/wuachumen/autosemilla-.git"
BRANCH="consolidated/vqcc-production"
WORK=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO=WORK/"autosemilla_repo"; STATE=WORK/"autosemilla_state"; STATE.mkdir(parents=True,exist_ok=True)
RUN=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT={"run_id":RUN,"utc":datetime.now(timezone.utc).isoformat(),"python":sys.version,"platform":platform.platform(),"stages":{},"official_verification":"PENDING"}
if not (REPO/"autosemilla"/"core.py").exists():
    if REPO.exists(): shutil.rmtree(REPO)
    p=subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,str(REPO)],capture_output=True,text=True,timeout=180)
    if p.returncode: raise RuntimeError("No se pudo clonar la rama. Activa Internet en Kaggle. "+p.stderr[-2000:])
sys.path.insert(0,str(REPO))
REPORT["stages"]["repository"]={"status":"PASS","commit":subprocess.run(["git","-C",str(REPO),"rev-parse","HEAD"],capture_output=True,text=True).stdout.strip()}
print(REPORT["stages"]["repository"])

In [ ]:
p=subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=REPO,capture_output=True,text=True,timeout=180)
REPORT["stages"]["unit_tests"]={"status":"PASS" if p.returncode==0 else "FAIL","stdout":p.stdout[-10000:],"stderr":p.stderr[-4000:]}
print(p.stdout,p.stderr[-2000:])
if p.returncode: raise RuntimeError("Fallaron las pruebas unitarias.")

In [ ]:
from autosemilla.core import Ledger,AdaptiveSearch,export_jsonl,import_jsonl,Seed,CheapEvaluator
from autosemilla.kickmix_ingest import ingest_kickmix_jsonl
ledger=Ledger(STATE/"ledger.sqlite"); search=AdaptiveSearch(ledger,seed=20261114)
smoke={}
for b in ("invsqrt","dqi","dfthcsos"):
    rows=search.run(b,24); promoted=search.promote(b,5)
    smoke[b]={"generated":len(rows),"valid":sum(e.valid for _,e in rows),"promoted":len(promoted),"verified_history":ledger.verified_count(b)}
    assert len(rows)==24 and smoke[b]["valid"]==24
REPORT["stages"]["search_smoke"]={"status":"PASS","benchmarks":smoke}
print(json.dumps(smoke,indent=2))

In [ ]:
checkpoint=STATE/"experiments.jsonl"; n=export_jsonl(ledger,checkpoint)
resume=Ledger(STATE/"resume_test.sqlite"); imported=import_jsonl(resume,checkpoint); duplicate=import_jsonl(resume,checkpoint)
assert imported==n and duplicate==0
REPORT["stages"]["persistence"]={"status":"PASS","records":n,"imported":imported,"duplicate_import":duplicate,"sha256":hashlib.sha256(checkpoint.read_bytes()).hexdigest()}
# Isolated synthetic adapter check: deliberately not counted as official verification.
with tempfile.TemporaryDirectory() as td:
    testdb=Ledger(Path(td)/"test.sqlite"); s=Seed("invsqrt","adapter-test",{"bits":64,"precision":32,"ancilla":4,"layers":4},7)
    testdb.add(s,CheapEvaluator().evaluate(s)); f=Path(td)/"kickmix_results.jsonl"
    f.write_text(json.dumps({"seed_id":s.id,"compiled":True,"verified":True,"verifier_status":"PASS","ranking_value":1.0})+"\n")
    counts=ingest_kickmix_jsonl(testdb,f); assert counts["verified"]==1
REPORT["stages"]["adapter_test"]={"status":"PASS","synthetic_only":True}

In [ ]:
root=Path("/kaggle/input"); inventory=[]
if root.exists():
    for d,dirs,files in os.walk(root):
        depth=len(Path(d).relative_to(root).parts); dirs[:]=dirs[:30] if depth<5 else []
        for name in files:
            p=Path(d)/name; inventory.append({"path":str(p),"bytes":p.stat().st_size})
            if len(inventory)>=10000: break
        if len(inventory)>=10000: break
sample=[x["path"] for x in inventory if Path(x["path"]).name.lower() in {"invsqrt.kmb","dqi.kmb","sample_submission.zip"}]
verifiers=[x["path"] for x in inventory if "verifier" in x["path"].lower()]
kickmix=[x["path"] for x in inventory if "kickmix" in x["path"].lower()]
(STATE/"input_inventory.json").write_text(json.dumps(inventory,indent=2))
REPORT["stages"]["official_inputs"]={"status":"READY_CANDIDATES_FOUND" if sample and verifiers else "BLOCKED_INPUTS","sample_candidates":sample[:20],"verifier_candidates":verifiers[:20],"kickmix_candidates":kickmix[:20]}
REPORT["final_ledger"]={"records":ledger.count(),"verified":ledger.verified_count(),"checkpoint":str(checkpoint)}
REPORT["official_verification"]="PENDING"
REPORT["readiness_note"]="Motor aprobado; la integración oficial exige inputs reales, candidato compilado y ejecución del verificador oficial. No se simula ese PASS."
out=WORK/f"autosemilla_report_{RUN}.json"; out.write_text(json.dumps(REPORT,indent=2,default=str))
print(json.dumps(REPORT,indent=2)); print("REPORT:",out); print("CHECKPOINT:",checkpoint)